# Notebook 3 – Market Trends Analysis
This notebook analyses the price gaps (actual – predicted) to identify market patterns, over‑/under‑priced segments, and temporal trends. It uses the predictions from the tuned RandomForest model.

# 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotting style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# 2. Load Predictions

In [ ]:
# Load the test predictions saved from the price prediction notebook
test_results = pd.read_csv('test_predictions.csv')

# Check the data
print(test_results.head())
print(test_results.shape)

# 3. Aggregate Trends by City (ville)

In [ ]:
# Group by ville
city_stats = test_results.groupby('ville').agg(
    avg_actual=('actual_price', 'mean'),
    avg_pred=('predicted_price', 'mean'),
    avg_gap=('price_gap', 'mean'),
    avg_gap_pct=('price_gap_pct', 'mean'),
    count=('actual_price', 'count')
).sort_values('count', ascending=False)

# Show top 10 cities by count
print(city_stats.head(10))

# 4. Visualise Top 10 Cities by Average Percentage Gap

In [ ]:
# Get top 10 cities with highest average positive gap (over‑priced)
top_overpriced = city_stats.sort_values('avg_gap_pct', ascending=False).head(10)
top_overpriced[['avg_gap_pct', 'count']].plot(kind='bar', figsize=(12,6))
plt.title('Top 10 Cities by Average Over‑valuation (%)')
plt.xlabel('City')
plt.ylabel('Average Gap (%)')
plt.tight_layout()
plt.show()

# Top 10 cities with highest average negative gap (under‑priced)
top_underpriced = city_stats.sort_values('avg_gap_pct').head(10)
top_underpriced[['avg_gap_pct', 'count']].plot(kind='bar', figsize=(12,6))
plt.title('Top 10 Cities by Average Under‑valuation (%)')
plt.xlabel('City')
plt.ylabel('Average Gap (%)')
plt.tight_layout()
plt.show()

# 5. Trends by Property Type and Contract

In [ ]:
type_contrat = test_results.groupby(['type', 'contrat']).agg(
    avg_gap_pct=('price_gap_pct', 'mean'),
    count=('actual_price', 'count')
).reset_index()

# Pivot for heatmap
pivot = type_contrat.pivot(index='type', columns='contrat', values='avg_gap_pct')
plt.figure(figsize=(8,6))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=0)
plt.title('Average Price Gap (%) by Property Type and Contract')
plt.tight_layout()
plt.show()

# 6. Impact of Amenities

In [ ]:
# List of binary features (add more if needed)
amenities = ['has_climatisation', 'has_chaffage', 'has_parking', 'has_jardin',
             'has_terrasse', 'has_balcon', 'has_ascenseur']

# For each amenity, compute average gap when present vs absent
for col in amenities:
    present = test_results[test_results[col] == 1]['price_gap_pct']
    absent = test_results[test_results[col] == 0]['price_gap_pct']
    print(f"{col}: present avg gap = {present.mean():.2f}%, absent avg gap = {absent.mean():.2f}%")

# 7. Price Gap by Surface Bins

In [ ]:
# Create surface bins (adjust bins as needed)
bins = [0, 50, 100, 150, 200, 500, np.inf]
labels = ['<50', '50-100', '100-150', '150-200', '200-500', '500+']
test_results['surface_bin'] = pd.cut(test_results['surface'], bins=bins, labels=labels)

surface_stats = test_results.groupby('surface_bin').agg(
    avg_gap_pct=('price_gap_pct', 'mean'),
    count=('actual_price', 'count')
)

surface_stats.plot(kind='bar', y='avg_gap_pct', figsize=(10,6))
plt.title('Average Price Gap by Surface Range')
plt.xlabel('Surface (sqm)')
plt.ylabel('Average Gap (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 8. Temporal Trends

In [ ]:
# Convert date_publication to datetime if present
if 'date_publication' in test_results.columns:
    test_results['date'] = pd.to_datetime(test_results['date_publication'], errors='coerce')
    test_results['year_month'] = test_results['date'].dt.to_period('M')
    time_stats = test_results.groupby('year_month').agg(
        avg_gap_pct=('price_gap_pct', 'mean'),
        count=('actual_price', 'count')
    ).dropna()

    # Plot
    time_stats['avg_gap_pct'].plot(figsize=(12,6))
    plt.title('Average Price Gap Over Time')
    plt.xlabel('Month')
    plt.ylabel('Average Gap (%)')
    plt.tight_layout()
    plt.show()
else:
    print("Date column not available; skipping time trend analysis.")

# 9. Identify Extreme Outliers (Mispriced Properties)

In [ ]:
# Over‑priced properties (price > predicted by more than 50%)
overpriced = test_results[test_results['price_gap_pct'] > 50].sort_values('price_gap_pct', ascending=False)
print(f"Number of overpriced properties (>50% over): {len(overpriced)}")
print(overpriced[['surface', 'type', 'ville', 'actual_price', 'predicted_price', 'price_gap_pct']].head(10))

# Under‑priced properties (price < predicted by more than 50%)
underpriced = test_results[test_results['price_gap_pct'] < -50].sort_values('price_gap_pct')
print(f"\nNumber of underpriced properties (< -50% under): {len(underpriced)}")
print(underpriced[['surface', 'type', 'ville', 'actual_price', 'predicted_price', 'price_gap_pct']].head(10))

# 10. Export Aggregated Results

In [ ]:
city_stats.to_csv('city_stats.csv')
type_contrat.to_csv('type_contrat_stats.csv')
surface_stats.to_csv('surface_stats.csv')
print("Aggregated results saved.")